In [1]:
# A small curated knowledge base of sepsis clinical guidelines
# Each entry is a short paragraph summarizing key clinical knowledge

guideline_documents = [
    "Elevated lactate levels above 2 mmol/L indicate tissue hypoperfusion and are a key marker of sepsis severity. Rising lactate over serial measurements is associated with worse outcomes and should prompt urgent reassessment of the patient's hemodynamic status.",
    
    "A Mean Arterial Pressure (MAP) below 65 mmHg indicates inadequate organ perfusion and is a criterion for septic shock when combined with elevated lactate. Vasopressor support should be considered when fluid resuscitation fails to restore MAP above this threshold.",
    
    "Tachycardia, defined as a heart rate greater than 90 beats per minute, is a common early compensatory response to infection as the body attempts to maintain cardiac output despite falling blood pressure. Persistent tachycardia despite fluid resuscitation may indicate ongoing hypovolemia or shock.",
    
    "The quick Sequential Organ Failure Assessment (qSOFA) score uses three criteria: respiratory rate of 22 or higher, altered mental status, and systolic blood pressure of 100 mmHg or lower. A qSOFA score of 2 or more identifies patients at higher risk of poor outcomes from sepsis.",
    
    "The Surviving Sepsis Campaign recommends initiating broad-spectrum antibiotics within one hour of sepsis recognition. Delayed antibiotic administration is associated with a measurable increase in mortality for each hour of delay.",
    
    "Systemic Inflammatory Response Syndrome (SIRS) criteria include heart rate above 90, respiratory rate above 20, temperature above 38C or below 36C, and abnormal white blood cell count. Meeting two or more SIRS criteria in the presence of suspected infection raises concern for sepsis.",
    
    "Fever (temperature above 38C) or hypothermia (temperature below 36C) both indicate a dysregulated immune response to infection. Hypothermia in the setting of infection is a particularly concerning sign, as it may indicate an inadequate immune response and is associated with higher mortality than fever.",
    
    "Tachypnea, or an elevated respiratory rate above 22 breaths per minute, often reflects the body's attempt to compensate for metabolic acidosis caused by tissue hypoxia in early sepsis. It is one of the earliest and most sensitive signs of clinical deterioration.",
    
    "The Shock Index, calculated as heart rate divided by systolic blood pressure, is a simple bedside tool for early identification of hemodynamic compromise. A Shock Index greater than 0.9 has been associated with increased risk of adverse outcomes in sepsis patients.",
    
    "Early goal-directed therapy for sepsis emphasizes rapid intravenous fluid resuscitation, typically 30 mL/kg of crystalloid within the first three hours, followed by frequent reassessment of the patient's hemodynamic response to guide further treatment.",
]

print(len(guideline_documents), "guideline documents loaded")

10 guideline documents loaded


In [2]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

# Load a pre-trained sentence embedding model (small, fast, good quality)
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Convert all guideline documents into embeddings (numbers representing their meaning)
document_embeddings = embedder.encode(guideline_documents, normalize_embeddings=True)

print("Embeddings shape:", document_embeddings.shape)

# Create a FAISS index for fast similarity search
# IndexFlatIP does "inner product" search, which works well with normalized embeddings for cosine similarity
index = faiss.IndexFlatIP(document_embeddings.shape[1])
index.add(np.array(document_embeddings))

print("FAISS index created with", index.ntotal, "documents")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\sahil\Downloads\ai_in_healthcare_project\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sahil\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings shape: (10, 384)
FAISS index created with 10 documents


In [3]:
# Function to search our guideline knowledge base for the most relevant documents
def retrieve_relevant_guidelines(query, top_k=3):
    # Convert the query text into an embedding (same way we did for documents)
    query_embedding = embedder.encode([query], normalize_embeddings=True)
    
    # Search the FAISS index for the most similar documents
    scores, indices = index.search(np.array(query_embedding), top_k)
    
    # Retrieve the actual document text for the top matches
    results = []
    for idx, score in zip(indices[0], scores[0]):
        results.append({
            "document": guideline_documents[idx],
            "similarity_score": float(score)
        })
    
    return results

print("Retrieval function ready")

Retrieval function ready


In [4]:
# Simulate a scenario: our model flagged high risk, and SHAP showed these were the top contributing features
test_query = "rising lactate levels, falling mean arterial pressure, high heart rate"

# Retrieve the most relevant guidelines for this scenario
results = retrieve_relevant_guidelines(test_query, top_k=3)

# Print the results
for i, r in enumerate(results):
    print(f"\n--- Match {i+1} (similarity: {r['similarity_score']:.3f}) ---")
    print(r['document'])


--- Match 1 (similarity: 0.561) ---
Elevated lactate levels above 2 mmol/L indicate tissue hypoperfusion and are a key marker of sepsis severity. Rising lactate over serial measurements is associated with worse outcomes and should prompt urgent reassessment of the patient's hemodynamic status.

--- Match 2 (similarity: 0.489) ---
A Mean Arterial Pressure (MAP) below 65 mmHg indicates inadequate organ perfusion and is a criterion for septic shock when combined with elevated lactate. Vasopressor support should be considered when fluid resuscitation fails to restore MAP above this threshold.

--- Match 3 (similarity: 0.453) ---
Tachycardia, defined as a heart rate greater than 90 beats per minute, is a common early compensatory response to infection as the body attempts to maintain cardiac output despite falling blood pressure. Persistent tachycardia despite fluid resuscitation may indicate ongoing hypovolemia or shock.


In [5]:
# Test with a different clinical scenario - focused on breathing/qSOFA related symptoms
test_query_2 = "elevated respiratory rate, low systolic blood pressure, altered mental status"

results_2 = retrieve_relevant_guidelines(test_query_2, top_k=3)

for i, r in enumerate(results_2):
    print(f"\n--- Match {i+1} (similarity: {r['similarity_score']:.3f}) ---")
    print(r['document'])


--- Match 1 (similarity: 0.508) ---
Tachypnea, or an elevated respiratory rate above 22 breaths per minute, often reflects the body's attempt to compensate for metabolic acidosis caused by tissue hypoxia in early sepsis. It is one of the earliest and most sensitive signs of clinical deterioration.

--- Match 2 (similarity: 0.399) ---
A Mean Arterial Pressure (MAP) below 65 mmHg indicates inadequate organ perfusion and is a criterion for septic shock when combined with elevated lactate. Vasopressor support should be considered when fluid resuscitation fails to restore MAP above this threshold.

--- Match 3 (similarity: 0.392) ---
The Shock Index, calculated as heart rate divided by systolic blood pressure, is a simple bedside tool for early identification of hemodynamic compromise. A Shock Index greater than 0.9 has been associated with increased risk of adverse outcomes in sepsis patients.


In [6]:
# Function to generate a clinical explanation from retrieved evidence
def generate_explanation(risk_score, query, retrieved_docs):
    # Start building the explanation
    explanation = f"Sepsis Risk Assessment: {risk_score*100:.1f}%\n\n"
    explanation += f"Triggering factors: {query}\n\n"
    explanation += "Clinical Evidence:\n"
    
    # Add each retrieved document as numbered evidence with citation
    for i, doc in enumerate(retrieved_docs):
        explanation += f"[{i+1}] {doc['document']}\n\n"
    
    # Build a summary that references the citations
    explanation += "Summary: "
    explanation += f"This patient's elevated risk is supported by clinical evidence "
    explanation += f"regarding {len(retrieved_docs)} relevant factors, as cited above [1"
    if len(retrieved_docs) > 1:
        explanation += f"-{len(retrieved_docs)}"
    explanation += "]. Clinical correlation and physician judgment are recommended."
    
    return explanation

# Test it with our first scenario
test_explanation = generate_explanation(0.78, test_query, results)
print(test_explanation)

Sepsis Risk Assessment: 78.0%

Triggering factors: rising lactate levels, falling mean arterial pressure, high heart rate

Clinical Evidence:
[1] Elevated lactate levels above 2 mmol/L indicate tissue hypoperfusion and are a key marker of sepsis severity. Rising lactate over serial measurements is associated with worse outcomes and should prompt urgent reassessment of the patient's hemodynamic status.

[2] A Mean Arterial Pressure (MAP) below 65 mmHg indicates inadequate organ perfusion and is a criterion for septic shock when combined with elevated lactate. Vasopressor support should be considered when fluid resuscitation fails to restore MAP above this threshold.

[3] Tachycardia, defined as a heart rate greater than 90 beats per minute, is a common early compensatory response to infection as the body attempts to maintain cardiac output despite falling blood pressure. Persistent tachycardia despite fluid resuscitation may indicate ongoing hypovolemia or shock.

Summary: This patient'

In [7]:
import pickle

# Save the FAISS index and documents for later use (e.g., in the final demo)
faiss.write_index(index, "../models/guideline_index.faiss")

with open("../models/guideline_documents.pkl", "wb") as f:
    pickle.dump(guideline_documents, f)

print("RAG pipeline components saved!")

RAG pipeline components saved!


In [8]:
# Import the NLI pipeline from transformers library
from transformers import pipeline

# Load a pre-trained NLI model (checks if one statement supports/contradicts another)
nli_model = pipeline("text-classification", model="roberta-large-mnli")

print("NLI model loaded")

config.json:   0%|          | 0.00/688 [00:00<?, ?B/s]

c:\Users\sahil\Downloads\ai_in_healthcare_project\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sahil\.cache\huggingface\hub\models--roberta-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.43GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

NLI model loaded


In [9]:
# Function to check if a claim is supported by (faithful to) a source document
def check_faithfulness(claim, source_document):
    # NLI models expect a "premise" (the source) and "hypothesis" (the claim to check)
    result = nli_model(f"{source_document} </s></s> {claim}")
    
    return result

# Test: check if our generated summary claim is supported by the lactate document
claim_to_check = "This patient's elevated risk is supported by evidence regarding rising lactate levels"
source_doc = guideline_documents[0]  # the lactate guideline

result = check_faithfulness(claim_to_check, source_doc)
print("Claim:", claim_to_check)
print("Source:", source_doc[:100], "...")
print("NLI Result:", result)

Claim: This patient's elevated risk is supported by evidence regarding rising lactate levels
Source: Elevated lactate levels above 2 mmol/L indicate tissue hypoperfusion and are a key marker of sepsis  ...
NLI Result: [{'label': 'ENTAILMENT', 'score': 0.660545289516449}]


In [10]:
# Function to check faithfulness of an entire generated explanation against its cited sources
def evaluate_explanation_faithfulness(retrieved_docs):
    faithfulness_results = []
    
    for i, doc_info in enumerate(retrieved_docs):
        source = doc_info['document']
        
        # Create a simple claim that represents "this evidence is relevant to the patient's risk"
        claim = f"This evidence supports the patient's elevated sepsis risk assessment."
        
        result = check_faithfulness(claim, source)
        label = result[0]['label']
        score = result[0]['score']
        
        faithfulness_results.append({
            "citation_number": i+1,
            "source": source[:80] + "...",
            "label": label,
            "confidence": score
        })
    
    return faithfulness_results

# Run faithfulness check on our test scenario's citations
faithfulness_check = evaluate_explanation_faithfulness(results)

for fc in faithfulness_check:
    print(f"[{fc['citation_number']}] Label: {fc['label']}, Confidence: {fc['confidence']:.3f}")
    print(f"    Source: {fc['source']}\n")

[1] Label: NEUTRAL, Confidence: 0.756
    Source: Elevated lactate levels above 2 mmol/L indicate tissue hypoperfusion and are a k...

[2] Label: NEUTRAL, Confidence: 0.736
    Source: A Mean Arterial Pressure (MAP) below 65 mmHg indicates inadequate organ perfusio...

[3] Label: NEUTRAL, Confidence: 0.907
    Source: Tachycardia, defined as a heart rate greater than 90 beats per minute, is a comm...



In [11]:
# Create specific claims that match what's actually in each source document (more realistic test)
specific_claims_and_sources = [
    ("The patient's rising lactate levels indicate tissue hypoperfusion and worsening sepsis severity.", guideline_documents[0]),
    ("The patient's falling mean arterial pressure indicates inadequate organ perfusion.", guideline_documents[1]),
    ("The patient's elevated heart rate reflects a compensatory response to infection.", guideline_documents[2]),
]

print("=== Faithfulness Check with Specific Claims ===\n")
for claim, source in specific_claims_and_sources:
    result = check_faithfulness(claim, source)
    label = result[0]['label']
    score = result[0]['score']
    print(f"Claim: {claim}")
    print(f"Result: {label} (confidence: {score:.3f})\n")

=== Faithfulness Check with Specific Claims ===

Claim: The patient's rising lactate levels indicate tissue hypoperfusion and worsening sepsis severity.
Result: ENTAILMENT (confidence: 0.971)

Claim: The patient's falling mean arterial pressure indicates inadequate organ perfusion.
Result: ENTAILMENT (confidence: 0.964)

Claim: The patient's elevated heart rate reflects a compensatory response to infection.
Result: ENTAILMENT (confidence: 0.975)



In [12]:
import json

# Compile faithfulness results
faithfulness_summary = {
    "generic_claims_test": {
        "avg_confidence": 0.80,  # average of the 3 NEUTRAL results
        "labels": ["NEUTRAL", "NEUTRAL", "NEUTRAL"]
    },
    "specific_claims_test": {
        "avg_confidence": (0.971 + 0.964 + 0.975) / 3,
        "labels": ["ENTAILMENT", "ENTAILMENT", "ENTAILMENT"]
    },
    "conclusion": "Specific, feature-grounded explanations achieve high faithfulness (avg 97%), while generic claims do not - informing our explanation generation design"
}

print("Average faithfulness confidence (specific claims):", faithfulness_summary["specific_claims_test"]["avg_confidence"])

with open("../reports/faithfulness_results.json", "w") as f:
    json.dump(faithfulness_summary, f, indent=2)

print("Faithfulness results saved!")

Average faithfulness confidence (specific claims): 0.9700000000000001
Faithfulness results saved!
